# Assignment 10 — Image Segmentation III and Image Compression I

Manual Canny edge detection and Hough line transform implementations.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import deque

LAB_DIR = Path.cwd()
def read_gray(name):
    image=cv2.imread(str(LAB_DIR/name),cv2.IMREAD_GRAYSCALE)
    if image is None: raise FileNotFoundError(LAB_DIR/name)
    return image

def conv(image,kernel,mode="edge"):
    p=kernel.shape[0]//2; pad=np.pad(image.astype(float),p,mode=mode); out=np.zeros(image.shape,float)
    for r in range(image.shape[0]):
        for c in range(image.shape[1]): out[r,c]=(pad[r:r+2*p+1,c:c+2*p+1]*kernel).sum()
    return out

def gaussian_kernel(size=5,sigma=1):
    axis=np.arange(-(size//2),size//2+1); xx,yy=np.meshgrid(axis,axis); k=np.exp(-(xx*xx+yy*yy)/(2*sigma*sigma)); return k/k.sum()

def canny_manual(image, low=100, high=200):
    smooth=conv(image,gaussian_kernel()); sx=np.array([[-1,0,1],[-2,0,2],[-1,0,1]]); sy=sx.T; gx=conv(smooth,sx); gy=conv(smooth,sy); mag=np.hypot(gx,gy); angle=(np.degrees(np.arctan2(gy,gx))+180)%180
    nms=np.zeros_like(mag)
    for r in range(1,mag.shape[0]-1):
        for c in range(1,mag.shape[1]-1):
            a=angle[r,c]; q1,q2=(mag[r,c-1],mag[r,c+1]) if a<22.5 or a>=157.5 else ((mag[r-1,c+1],mag[r+1,c-1]) if a<67.5 else ((mag[r-1,c],mag[r+1,c]) if a<112.5 else (mag[r-1,c-1],mag[r+1,c+1])))
            if mag[r,c]>=q1 and mag[r,c]>=q2: nms[r,c]=mag[r,c]
    strong=nms>=high; weak=(nms>=low)&~strong; result=strong.copy(); queue=deque(map(tuple,np.argwhere(strong)))
    while queue:
        r,c=queue.popleft()
        for dr in (-1,0,1):
            for dc in (-1,0,1):
                nr,nc=r+dr,c+dc
                if 0<=nr<result.shape[0] and 0<=nc<result.shape[1] and weak[nr,nc] and not result[nr,nc]: result[nr,nc]=True; queue.append((nr,nc))
    return smooth,mag,nms,(result*255).astype(np.uint8)

messi=read_gray("messi.png"); smooth, magnitude, suppressed, edges=canny_manual(messi)
fig,ax=plt.subplots(1,5,figsize=(18,4));
for a,b,t in zip(ax,[messi,smooth,magnitude,suppressed,edges],["Original","Gaussian","Gradient","NMS","Canny"]): a.imshow(b,cmap="gray"); a.set_title(t); a.axis("off")
plt.show()

def hough_lines(image, threshold=100):
    ys,xs=np.nonzero(image); diag=int(np.ceil(np.hypot(*image.shape))); thetas=np.deg2rad(np.arange(180)); rhos=np.arange(-diag,diag+1); acc=np.zeros((len(rhos),len(thetas)),np.uint32)
    cos,sin=np.cos(thetas),np.sin(thetas)
    for x,y in zip(xs,ys):
        indices=np.rint(x*cos+y*sin).astype(int)+diag
        acc[indices,np.arange(180)]+=1
    lines=[]
    for ri,ti in zip(*np.where(acc>=threshold)): lines.append((rhos[ri],np.rad2deg(thetas[ti]),int(acc[ri,ti])))
    return acc,lines

sudoku=read_gray("sudoku.png"); _,_,_,sudoku_edges=canny_manual(sudoku,50,150); accumulator, lines=hough_lines(sudoku_edges,100)
print("Detected Hough lines:", len(lines)); print(lines[:20])
plt.figure(figsize=(8,5)); plt.imshow(sudoku_edges,cmap="gray"); plt.title("Sudoku Canny edges"); plt.axis("off"); plt.show()
color=cv2.cvtColor(sudoku,cv2.COLOR_GRAY2BGR)
for rho,theta,_ in lines:
    t=np.deg2rad(theta); a,b=np.cos(t),np.sin(t); x0,y0=a*rho,b*rho; p1=(int(x0+2000*(-b)),int(y0+2000*a)); p2=(int(x0-2000*(-b)),int(y0-2000*a)); cv2.line(color,p1,p2,(0,0,255),1)
plt.figure(figsize=(8,5)); plt.imshow(cv2.cvtColor(color,cv2.COLOR_BGR2RGB)); plt.title("Detected lines"); plt.axis("off"); plt.show()

: 